# [skip-ci]

# Quickstart: from FASTQ to per-cell viral load

**Utility:** ViralScan turns paired-end 10x scRNA-seq FASTQs into per-cell viral UMI
quantification with one command. This notebook runs the whole path on a small public
sample and reads the key outputs.

Marked `[skip-ci]`: it builds a kallisto index and runs `kb count`, which are too
heavy for CI. Requires `viralscan >= 2.3.0`, plus `kb` (kb-python) and `snakemake` on
PATH. Download helpers (`sra-tools`) are only for fetching the public FASTQ.

## Step 0 — preflight: are the engines installed?

In [ ]:
!viralscan --version
!kb --version
!snakemake --version

## Step 1 — get a small public sample

We use a **1M-read subsample** of the EBV LCL sample `SRR12682296` (10x Chromium v2)
— the same sample the manuscript uses for the multimapping comparison. Full depth is
~112M reads; the subsample keeps the tutorial laptop-scale.

Fetch from ENA/SRA (no institutional paths — edit `WORKDIR` to any local directory):

In [ ]:
import os, subprocess
from pathlib import Path

WORKDIR = Path(os.environ.get('VS_WORKDIR', 'viralscan_quickstart')).resolve()
fastq_dir = WORKDIR / 'fastq'
fastq_dir.mkdir(parents=True, exist_ok=True)
SRR = 'SRR12682296'

# Download (SRA), then subsample to 1M reads with seqkit (or use `fastq-dump -X 1000000`).
# Shown as commands; run them in a shell with sra-tools + seqkit available:
print(f'''
  prefetch {SRR} --output-directory {WORKDIR}/sra
  fasterq-dump {WORKDIR}/sra/{SRR} --split-files --outdir {fastq_dir}
  seqkit head -n 1000000 {fastq_dir}/{SRR}_1.fastq > {fastq_dir}/{SRR}_1.sub.fastq
  seqkit head -n 1000000 {fastq_dir}/{SRR}_2.fastq > {fastq_dir}/{SRR}_2.sub.fastq
''')
r1 = fastq_dir / f'{SRR}_1.sub.fastq'
r2 = fastq_dir / f'{SRR}_2.sub.fastq'  # R2 is the (longer) cDNA read

## Step 2 — chemistry preflight (`check-whitelist`)

Before a full run, confirm the R1 barcodes match the whitelist for your assumed
chemistry. A low match rate means the wrong `--technology` (this is exactly what
flagged the COVID GEM-X libraries at 0.4% in the paper). See the **specificity**
vignette for a runnable demo of this check.

In [ ]:
# 10x v2 whitelist ships with kb-python / cellranger; point -w at it.
!viralscan check-whitelist -s1 {r1} -w 10x_v2_whitelist.txt -x 10xv2 --min-match-rate 0.5

## Step 3 — quantify

Provide a prebuilt combined host+virus index (`-i`/`-t`; see the **reference**
vignette to build one). ViralScan writes results under `OUTPUT/<sample>/`.

In [ ]:
OUTPUT = WORKDIR / 'out'
cmd = [
    'viralscan',
    '-i', 'ref/index.idx', '-t', 'ref/t2g.txt',
    '-o', str(OUTPUT),
    '-s1', str(r1), '-s2', str(r2),
    '-x', '10xv2',
    '--multimap-method', 'em',   # EM multimapping correction (see multimapping vignette)
    '--cell-calling', 'knee',    # denominator of real cells (see cell-calling vignette)
    '--cores', '8',
]
print(' '.join(cmd))
# subprocess.run(cmd, check=True)

## Step 4 — read the viral summary

`results/viral_summary.tsv` has one row per detected virus. Key columns: `total_umi`,
`infected_cells`, `pct_infected` (all barcodes), and **`pct_infected_called`** (over
real cells — the primary rate). Full schema: `docs/output_reference.md`.

In [ ]:
import pandas as pd
sample_dir = OUTPUT / SRR
summary = sample_dir / 'results' / 'viral_summary.tsv'
if summary.exists():
    df = pd.read_csv(summary, sep='\t')
    display(df.sort_values('total_umi', ascending=False).head(10))
else:
    print('Run Step 3 first. Expected columns:')
    print('virus_name, total_umi, infected_cells, total_cells, pct_infected,')
    print('umi_per_10k, n_called_cells, infected_called, pct_infected_called, ...')

## Step 5 — per-cell table and the HTML report

In [ ]:
per_cell = sample_dir / 'results' / 'per_cell_viral.tsv'
report = sample_dir / 'report.html'
if per_cell.exists():
    pc = pd.read_csv(per_cell, sep='\t')
    print(f'{pc.barcode.nunique()} infected cells across {pc.virus_name.nunique()} viruses')
    display(pc.head())
print('Open the self-contained report:', report)

## Summary

One `viralscan` call took raw FASTQs to per-cell viral load. From here:

- **multimapping_correction** — why `--multimap-method em` recovers more signal.
- **cell_calling_denominators** — reporting rates over real cells.
- **cell_type_enrichment** — which cell types are infected.
- **qc_and_read_evidence** — confirming a call is real.